# CAG (Cache-Augmented Generation) Example 01-1: Document Registration

このノートブックでは、CAGアプローチの第一段階として、backlog-wikis配下の全ドキュメントをRedisキャッシュに保存します。

## 処理の流れ
1. backlog-wikisディレクトリ内の全Markdownファイルを再帰的に読み込み
2. 各ドキュメントをチャンクに分割
3. チャンクからembeddingを生成
4. よくある質問とその回答をLLMで生成
5. 全てのデータをRedisキャッシュに保存

## 必要なライブラリのインポート

In [ ]:
import os
import json
import hashlib
import glob
from datetime import datetime
import redis
from langchain_ollama import OllamaLLM
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
import pandas as pd
from tqdm import tqdm
import time

## 設定の定義

In [ ]:
# Ollama設定
OLLAMA_BASE_URL = 'http://llm-rag-examples-ollama:11434'
# OLLAMA_MODEL = 'llama3.1:8b'
OLLAMA_MODEL = 'yuiseki/tinyswallow:1.5b'

# Redis設定
REDIS_HOST = 'llm-rag-examples-redis'
REDIS_PORT = 6379
REDIS_DB = 0

# ドキュメントの設定
DOCUMENT_BASE_PATH = './backlog-wikis'

# Embedding設定
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'

# チャンク設定
CHUNK_SIZE = 200
CHUNK_OVERLAP = 0

print(f"Ollama Base URL: {OLLAMA_BASE_URL}")
print(f"Ollama Model: {OLLAMA_MODEL}")
print(f"Document Base Path: {DOCUMENT_BASE_PATH}")
print(f"Embedding Model: {EMBEDDING_MODEL}")
print(f"Chunk Size: {CHUNK_SIZE}")

## 全ドキュメントファイルの発見

In [ ]:
# backlog-wikis配下の全Markdownファイルを再帰的に発見
markdown_files = glob.glob(os.path.join(DOCUMENT_BASE_PATH, '**', '*.md'), recursive=True)

print(f"Found {len(markdown_files)} Markdown files")
print("\n=== First 10 Files ===")
for i, file_path in enumerate(markdown_files[:10]):
    relative_path = os.path.relpath(file_path, DOCUMENT_BASE_PATH)
    print(f"{i+1:2d}. {relative_path}")

if len(markdown_files) > 10:
    print(f"... and {len(markdown_files) - 10} more files")

## Redisクライアントの初期化

In [ ]:
# Redisクライアントの初期化
try:
    redis_client = redis.Redis(
        host=REDIS_HOST,
        port=REDIS_PORT,
        db=REDIS_DB,
        decode_responses=True
    )
    # 接続テスト
    redis_client.ping()
    print("✓ Redis connection successful")
    use_redis = True
except Exception as e:
    print(f"✗ Redis connection failed: {e}")
    print("Using in-memory dictionary as fallback")
    redis_client = {}
    use_redis = False

print(f"Using Redis: {use_redis}")

In [ ]:
# 既存のCAGキャッシュを削除
print("Clearing existing CAG cache...")

try:
    if use_redis:
        # CAGプレフィックスを持つ全てのキーを削除
        cag_keys = redis_client.keys('cag*')
        if cag_keys:
            deleted_count = redis_client.delete(*cag_keys)
            print(f"✓ Deleted {deleted_count} existing CAG cache entries")
        else:
            print("✓ No existing CAG cache entries found")
    else:
        # インメモリ辞書の場合、CAGプレフィックスのキーを削除
        cag_keys = [k for k in redis_client.keys() if k.startswith('cag')]
        for key in cag_keys:
            del redis_client[key]
        print(f"✓ Deleted {len(cag_keys)} existing CAG cache entries from in-memory dictionary")
        
except Exception as e:
    print(f"⚠ Error clearing cache: {e}")
    print("Continuing with registration process...")

print("Cache cleared. Ready for new data registration.")

## 既存のCAGキャッシュをクリア

## Ollamaクライアントの初期化

In [ ]:
# Ollamaクライアントの初期化
ollama_llm = OllamaLLM(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0.3
)
print(f"✓ Ollama client initialized with model: {OLLAMA_MODEL}")
print(f"✓ Ollama base URL: {OLLAMA_BASE_URL}")

## Embeddingモデルの初期化

In [ ]:
# Embeddingモデルの初期化
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
print(f"✓ Embedding model loaded: {EMBEDDING_MODEL}")

## 全ドキュメントの読み込み

In [ ]:
# 全ドキュメントを読み込み
documents = []
total_characters = 0

print("Loading all documents...")
for file_path in tqdm(markdown_files, desc="Loading documents"):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
            relative_path = os.path.relpath(file_path, DOCUMENT_BASE_PATH)
            documents.append({
                'file_path': file_path,
                'relative_path': relative_path,
                'content': content,
                'characters': len(content)
            })
            total_characters += len(content)
    except Exception as e:
        print(f"\n✗ Error loading {file_path}: {e}")

print(f"\n✓ Loaded {len(documents)} documents")
print(f"✓ Total characters: {total_characters:,}")
print(f"✓ Average characters per document: {total_characters // len(documents):,}")

# ドキュメントサイズの統計
doc_sizes = [doc['characters'] for doc in documents]
print(f"\n=== Document Size Statistics ===")
print(f"Min size: {min(doc_sizes):,} characters")
print(f"Max size: {max(doc_sizes):,} characters")
print(f"Median size: {sorted(doc_sizes)[len(doc_sizes)//2]:,} characters")

## ドキュメントの結合とチャンク分割

In [ ]:
# chromadb-ex02.ipynbを参考にしたチャンク分割の設定
text_splitter = RecursiveCharacterTextSplitter(
    separators=[
        "$",
        "\n\n",
        "\uff0e",  # 全角「。」
        "\n",
        "\uff0c",  # 全角コンマ
        ".",
        ",",
        " ",
        "",
    ],
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

print("✓ Text splitter configured")

In [ ]:
# 各ドキュメントをチャンクに分割
all_chunks = []
document_chunk_map = []  # チャンクがどのドキュメントから来たかを記録

print("Splitting documents into chunks...")
for doc_idx, document in enumerate(tqdm(documents, desc="Chunking documents")):
    if document['content'].strip():  # 空でないコンテンツのみ処理
        chunks = text_splitter.split_text(document['content'])
        for chunk_idx, chunk in enumerate(chunks):
            all_chunks.append(chunk)
            document_chunk_map.append({
                'chunk_global_id': len(all_chunks) - 1,
                'chunk_local_id': chunk_idx,
                'document_id': doc_idx,
                'document_path': document['relative_path'],
                'chunk_text': chunk
            })

print(f"\n✓ Created {len(all_chunks)} chunks from {len(documents)} documents")
print(f"✓ Average chunks per document: {len(all_chunks) / len(documents):.1f}")

# チャンクサイズの統計
chunk_sizes = [len(chunk) for chunk in all_chunks]
print(f"\n=== Chunk Size Statistics ===")
print(f"Min size: {min(chunk_sizes)} characters")
print(f"Max size: {max(chunk_sizes)} characters")
print(f"Average size: {sum(chunk_sizes) / len(chunk_sizes):.1f} characters")

# チャンクのプレビュー
print("\n=== Chunk Preview ===")
for i in range(min(3, len(all_chunks))):
    chunk_info = document_chunk_map[i]
    print(f"\n--- Chunk {i+1} (from {chunk_info['document_path']}) ---")
    print(all_chunks[i][:150] + "..." if len(all_chunks[i]) > 150 else all_chunks[i])

## チャンクのembeddingを生成

In [ ]:
# 全チャンクのembeddingを生成
chunk_embeddings = []
print("Generating embeddings for chunks...")

# バッチ処理でembeddingを生成（メモリ効率を考慮）
batch_size = 50
for i in tqdm(range(0, len(all_chunks), batch_size), desc="Embedding chunks"):
    batch_chunks = all_chunks[i:i+batch_size]
    for chunk in batch_chunks:
        embedding = embeddings.embed_query(chunk)
        chunk_embeddings.append(embedding)

print(f"\n✓ Generated {len(chunk_embeddings)} embeddings")
print(f"✓ Embedding dimension: {len(chunk_embeddings[0])}")

## よくある質問の定義

In [ ]:
# よくある質問を定義（全社内連絡に関する一般的な質問）
common_questions = [
    "ブログの管理画面にはどうやってログインしますか？",
    "記事を投稿するときの注意点は何ですか？",
    "下書きが完了したら誰に連絡すればよいですか？",
    "WordPressのURLは何ですか？",
    "Wi-Fiのパスワードは何ですか？",
    "営業部の業務フローについて教えてください",
    "福利厚生にはどのようなものがありますか？",
    "施設予約の方法を教えてください",
    "ZOOMroomsの使い方を教えてください",
    "申請や備品関連の手続きについて教えてください",
    "Markdownの書き方を教えてください",
    "社内連絡の方法について教えてください"
]

print(f"Defined {len(common_questions)} common questions")
for i, q in enumerate(common_questions, 1):
    print(f"{i:2d}. {q}")

## LLMで質問の回答を生成

In [ ]:
# 質問と回答のペアを保存するリスト
qa_pairs = []

In [ ]:
print("Generating answers for common questions...")
print("(Using combined document content for context)")

# 全ドキュメントの内容を結合（コンテキストとして使用）
combined_content = "\n\n=== Document Separator ===\n\n".join([doc['content'] for doc in documents])
print(f"Combined content length: {len(combined_content):,} characters")

# コンテキストが長すぎる場合は制限
max_context_length = 50000  # 約50KB
if len(combined_content) > max_context_length:
    combined_content = combined_content[:max_context_length] + "\n\n[Content truncated due to length...]"
    print(f"Context truncated to {max_context_length:,} characters")

for i, question in enumerate(tqdm(common_questions, desc="Generating answers")):
    if len(qa_pairs) >= i + 1:
        continue

    # Ollamaを使用したプロンプトの構築
    prompt = f"""以下の社内文書を参考に、質問に回答してください。

社内文書:
{combined_content}

質問: {question}

回答は具体的で実用的な内容にしてください。文書に記載されていない内容については「文書に記載されていません」と回答してください。"""

    try:
        # Ollamaに回答生成をリクエスト
        print(f"\n=== Q&A {i+1} ===")
        print(f"Q: {question}")
        
        answer = ollama_llm.invoke(prompt)
        
        # Q&Aペアをリストに追加
        qa_pairs.append({
            'question': question,
            'answer': answer.strip(),
            'timestamp': datetime.now().isoformat()
        })
        
        print(f"A: {answer[:300]}..." if len(answer) > 300 else f"A: {answer}")
        
    except Exception as e:
        print(f"\n✗ Error generating answer for question {i+1}: {e}")
        qa_pairs.append({
            'question': question,
            'answer': "回答の生成に失敗しました。",
            'timestamp': datetime.now().isoformat()
        })
    
    # APIレート制限を避けるため少し待機
    time.sleep(5)

print(f"\n✓ Generated {len(qa_pairs)} Q&A pairs")

## 質問のembeddingを生成

In [ ]:
# 質問のembeddingを生成
print("Generating embeddings for questions...")

for i, qa_pair in enumerate(tqdm(qa_pairs, desc="Embedding questions")):
    question_embedding = embeddings.embed_query(qa_pair['question'])
    qa_pair['question_embedding'] = question_embedding

print(f"✓ Generated embeddings for {len(qa_pairs)} questions")

## データをRedisキャッシュに保存

In [ ]:
# チャンクデータをキャッシュに保存
print("Saving chunk data to cache...")

chunk_keys = []
for i, (chunk, embedding, chunk_info) in enumerate(tqdm(zip(all_chunks, chunk_embeddings, document_chunk_map), desc="Caching chunks")):
    # キャッシュキーを生成
    hash_value = hashlib.md5(chunk.encode('utf-8')).hexdigest()
    cache_key = f"cag_chunk:{hash_value}"
    chunk_keys.append(cache_key)
    
    # キャッシュデータを構築
    cache_data = {
        'type': 'chunk',
        'text': chunk,
        'embedding': embedding,
        'chunk_global_id': chunk_info['chunk_global_id'],
        'chunk_local_id': chunk_info['chunk_local_id'],
        'document_id': chunk_info['document_id'],
        'document_path': chunk_info['document_path'],
        'timestamp': datetime.now().isoformat()
    }
    
    # キャッシュに保存
    if use_redis:
        redis_client.set(cache_key, json.dumps(cache_data, ensure_ascii=False))
    else:
        redis_client[cache_key] = json.dumps(cache_data, ensure_ascii=False)

print(f"✓ Cached {len(chunk_keys)} chunks")

In [ ]:
# Q&Aデータをキャッシュに保存
print("Saving Q&A data to cache...")

qa_keys = []
for qa_pair in tqdm(qa_pairs, desc="Caching Q&A pairs"):
    # キャッシュキーを生成
    hash_value = hashlib.md5(qa_pair['question'].encode('utf-8')).hexdigest()
    cache_key = f"cag_qa:{hash_value}"
    qa_keys.append(cache_key)
    
    # キャッシュデータを構築
    cache_data = {
        'type': 'qa_pair',
        'question': qa_pair['question'],
        'answer': qa_pair['answer'],
        'question_embedding': qa_pair['question_embedding'],
        'timestamp': qa_pair['timestamp']
    }
    
    # キャッシュに保存
    if use_redis:
        redis_client.set(cache_key, json.dumps(cache_data, ensure_ascii=False))
    else:
        redis_client[cache_key] = json.dumps(cache_data, ensure_ascii=False)

print(f"✓ Cached {len(qa_keys)} Q&A pairs")

## ドキュメントメタデータをキャッシュに保存

In [ ]:
# ドキュメントメタデータをキャッシュに保存
print("Saving document metadata to cache...")

# ドキュメントインデックスをキャッシュに保存
document_index = {
    'type': 'document_index',
    'total_documents': len(documents),
    'total_chunks': len(all_chunks),
    'total_characters': total_characters,
    'documents': [
        {
            'id': i,
            'path': doc['relative_path'],
            'characters': doc['characters']
        }
        for i, doc in enumerate(documents)
    ],
    'timestamp': datetime.now().isoformat()
}

if use_redis:
    redis_client.set('cag_meta:document_index', json.dumps(document_index, ensure_ascii=False))
else:
    redis_client['cag_meta:document_index'] = json.dumps(document_index, ensure_ascii=False)

print("✓ Cached document metadata")

## キャッシュの確認

In [ ]:
# キャッシュの統計情報を取得
if use_redis:
    all_keys = redis_client.keys('cag*')
    chunk_keys_count = len(redis_client.keys('cag_chunk:*'))
    qa_keys_count = len(redis_client.keys('cag_qa:*'))
    meta_keys_count = len(redis_client.keys('cag_meta:*'))
else:
    all_keys = list(redis_client.keys())
    chunk_keys_count = len([k for k in all_keys if k.startswith('cag_chunk:')])
    qa_keys_count = len([k for k in all_keys if k.startswith('cag_qa:')])
    meta_keys_count = len([k for k in all_keys if k.startswith('cag_meta:')])

print("=== Cache Statistics ===")
print(f"Total keys: {len(all_keys)}")
print(f"Chunk keys: {chunk_keys_count}")
print(f"Q&A keys: {qa_keys_count}")
print(f"Metadata keys: {meta_keys_count}")
print(f"Cache storage: {'Redis' if use_redis else 'In-memory dictionary'}")

In [ ]:
# サンプルデータの確認
if chunk_keys:
    sample_chunk_key = chunk_keys[0]
    if use_redis:
        sample_chunk_data = json.loads(redis_client.get(sample_chunk_key))
    else:
        sample_chunk_data = json.loads(redis_client[sample_chunk_key])
    
    print(f"\n=== Sample Chunk Data ===")
    print(f"Type: {sample_chunk_data['type']}")
    print(f"Document Path: {sample_chunk_data['document_path']}")
    print(f"Chunk Global ID: {sample_chunk_data['chunk_global_id']}")
    print(f"Chunk Local ID: {sample_chunk_data['chunk_local_id']}")
    print(f"Text: {sample_chunk_data['text'][:100]}...")
    print(f"Embedding dimension: {len(sample_chunk_data['embedding'])}")

if qa_keys:
    sample_qa_key = qa_keys[0]
    if use_redis:
        sample_qa_data = json.loads(redis_client.get(sample_qa_key))
    else:
        sample_qa_data = json.loads(redis_client[sample_qa_key])
    
    print(f"\n=== Sample Q&A Data ===")
    print(f"Type: {sample_qa_data['type']}")
    print(f"Question: {sample_qa_data['question']}")
    print(f"Answer: {sample_qa_data['answer'][:100]}...")
    print(f"Question embedding dimension: {len(sample_qa_data['question_embedding'])}")

## 結果をCSVファイルに保存

In [ ]:
# Q&AデータをDataFrameに変換
qa_df = pd.DataFrame(qa_pairs)
qa_df_display = qa_df.drop('question_embedding', axis=1)  # embeddingは表示用から除外

print("=== Q&A Data Preview ===")
print(qa_df_display.head())

# CSVファイルに保存
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
qa_csv_filename = f'output/cag-examples01-1-qa-pairs.{timestamp}.csv'
qa_df_display.to_csv(qa_csv_filename, index=False)
print(f"\n✓ Q&A pairs saved to: {qa_csv_filename}")

# ドキュメントメタデータをCSVに保存
doc_df = pd.DataFrame([
    {
        'document_id': i,
        'relative_path': doc['relative_path'],
        'characters': doc['characters']
    }
    for i, doc in enumerate(documents)
])

doc_csv_filename = f'output/cag-examples01-1-documents.{timestamp}.csv'
doc_df.to_csv(doc_csv_filename, index=False)
print(f"✓ Document metadata saved to: {doc_csv_filename}")

# チャンクメタデータをCSVに保存
chunk_df = pd.DataFrame(document_chunk_map)
chunk_csv_filename = f'output/cag-examples01-1-chunks.{timestamp}.csv'
chunk_df.to_csv(chunk_csv_filename, index=False)
print(f"✓ Chunk metadata saved to: {chunk_csv_filename}")

## 登録完了サマリー

In [ ]:
print("\n" + "="*80)
print("CAG DOCUMENT REGISTRATION COMPLETED")
print("="*80)
print(f"✓ Source directory: {DOCUMENT_BASE_PATH}")
print(f"✓ Documents processed: {len(documents)}")
print(f"✓ Total characters: {total_characters:,}")
print(f"✓ Total chunks created: {len(all_chunks)}")
print(f"✓ Total Q&A pairs generated: {len(qa_pairs)}")
print(f"✓ Total cache entries: {len(all_keys)}")
print(f"✓ Cache storage: {'Redis' if use_redis else 'In-memory dictionary'}")
print(f"✓ Embedding model: {EMBEDDING_MODEL}")
print(f"✓ LLM model: {OLLAMA_MODEL}")
print(f"\n✓ Results saved to:")
print(f"  - Q&A pairs: {qa_csv_filename}")
print(f"  - Documents: {doc_csv_filename}")
print(f"  - Chunks: {chunk_csv_filename}")
print(f"\nNext step: Run 'cag-examples02-ollama-2-generate.ipynb' to use the cached data for question answering.")
print("="*80)